# ML Pipeline: From Graphs to Models

This notebook shows how to prepare graphs for machine learning training, including:
- Creating datasets from graphs
- Batching and collating
- Data augmentation
- Train/validation/test splits
- Integration with PyTorch

In [ ]:
import numpy as np
import weather_model_graphs as wmg
import matplotlib.pyplot as plt

print(f"weather-model-graphs: {wmg.__version__}")

## Part 1: Preparing Data

### Step 1a: Load or create graph with features

In [ ]:
# Load prebuilt graph
graph = wmg.load_prebuilt("keisler", grid_size=16)
print(f"Loaded graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# Add sample weather data
for node in graph.nodes():
    pos = graph.nodes[node].get("pos", np.array([0.5, 0.5]))
    # Synthetic weather variables
    graph.nodes[node]["temperature"] = 273.15 + 10 * np.sin(pos[0] * 2 * np.pi)
    graph.nodes[node]["u_wind"] = 5 + 3 * np.sin(pos[0] * np.pi)
    graph.nodes[node]["v_wind"] = 3 + 2 * np.cos(pos[1] * np.pi)
    graph.nodes[node]["pressure"] = 1000 + 20 * np.sin(pos[0] * np.pi) * np.cos(pos[1] * np.pi)

print(f"Added weather variables to nodes")
print(f"Sample node attributes: {list(graph.nodes[0].keys())}")

### Step 1b: Add engineered features

In [ ]:
# Add computed features
graph = wmg.add_wind_velocity(graph, u_attr="u_wind", v_attr="v_wind")
graph = wmg.add_wind_direction(graph, u_attr="u_wind", v_attr="v_wind")
graph = wmg.add_pressure_gradient(graph, pressure_attr="pressure")
graph = wmg.add_node_degree_features(graph)

print("Added engineered features:")
print("  - Wind velocity magnitude")
print("  - Wind direction")
print("  - Pressure gradient")
print("  - Node degree features")

# Normalize features
graph = wmg.normalize_features(
    graph,
    feature_keys=["temperature", "u_wind", "v_wind", "pressure", "wind_velocity"],
    method="zscore"
)
print("\nFeatures normalized with z-score")

## Part 2: Data Splits

### Node-level splits (for node prediction tasks)

In [ ]:
# Split nodes into train/val/test
train_nodes, val_nodes, test_nodes = wmg.split_graph_for_training(
    graph,
    train_size=0.7,
    val_size=0.15,
)

print("Node-level split:")
print(f"  Train: {len(train_nodes)} nodes ({len(train_nodes)/graph.number_of_nodes()*100:.1f}%)")
print(f"  Val:   {len(val_nodes)} nodes ({len(val_nodes)/graph.number_of_nodes()*100:.1f}%)")
print(f"  Test:  {len(test_nodes)} nodes ({len(test_nodes)/graph.number_of_nodes()*100:.1f}%)")

# Visualize split
fig, ax = plt.subplots(figsize=(8, 6))
splits = ["Train", "Val", "Test"]
sizes = [len(train_nodes), len(val_nodes), len(test_nodes)]
colors = ["#4CAF50", "#2196F3", "#FF9800"]
ax.bar(splits, sizes, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel("Number of Nodes")
ax.set_title("Train/Val/Test Node Split")
for i, (split, size) in enumerate(zip(splits, sizes)):
    ax.text(i, size + 5, f"{size}", ha='center', weight='bold')
plt.tight_layout()
plt.savefig('/tmp/node_split.png', dpi=150, bbox_inches='tight')
plt.show()

### Graph-level splits (for graph classification/regression)

In [ ]:
# Create multiple graphs (e.g., different timesteps or locations)
graph_list = [wmg.load_prebuilt("keisler", grid_size=16) for _ in range(20)]
print(f"Created dataset with {len(graph_list)} graphs")

# Add labels to graphs (e.g., weather regime classification)
for i, g in enumerate(graph_list):
    # Synthetic label: 0=stable, 1=unstable, 2=transition
    g.graph['label'] = i % 3
    g.graph['regime'] = ['stable', 'unstable', 'transition'][i % 3]

# Prepare splits
np.random.seed(42)
indices = np.random.permutation(len(graph_list))
n_train = int(0.7 * len(graph_list))
n_val = int(0.15 * len(graph_list))

train_graphs = [graph_list[i] for i in indices[:n_train]]
val_graphs = [graph_list[i] for i in indices[n_train:n_train+n_val]]
test_graphs = [graph_list[i] for i in indices[n_train+n_val:]]

print(f"\nGraph-level split:")
print(f"  Train: {len(train_graphs)} graphs")
print(f"  Val:   {len(val_graphs)} graphs")
print(f"  Test:  {len(test_graphs)} graphs")

# Show label distribution
regimes = [g.graph['regime'] for g in graph_list]
print(f"\nLabel distribution: {dict(zip(*np.unique(regimes, return_counts=True)))}")

## Part 3: Batching Graphs

In [ ]:
# Batch multiple graphs into a single large graph
node_features, edge_indices, edge_features = wmg.batch_graphs(
    train_graphs[:5],
    node_feature_keys=["pos"],
    edge_feature_keys=["len"],
)

print("Batched 5 graphs:")
print(f"  Node features shape: {node_features.shape}")
print(f"  Number of edge index arrays: {len(edge_indices)}")
if edge_features:
    if isinstance(edge_features, np.ndarray):
        print(f"  Edge features shape: {edge_features.shape}")
    else:
        print(f"  Edge features: list of {len(edge_features)} items")
else:
    print(f"  No edge features")

# Calculate combined size
total_nodes = sum(g.number_of_nodes() for g in train_graphs[:5])
total_edges = sum(g.number_of_edges() for g in train_graphs[:5])
print(f"\nCombined statistics:")
print(f"  Total nodes: {total_nodes}")
print(f"  Total edges: {total_edges}")

## Part 4: DataLoaders (if PyTorch is available)

In [ ]:
# Try creating DataLoader
try:
    # Create simple batches of graphs
    dataloader = wmg.create_dataloader(
        train_graphs,
        batch_size=4,
        shuffle=True,
        backend="networkx",
    )
    print(f"DataLoader created: {len(dataloader)} batches")
    print(f"Batch size: ~4 graphs per batch")
    
    # Try PyTorch Geometric if available
    try:
        pyg_loader = wmg.create_pyg_dataloader(
            train_graphs[:5],
            batch_size=2,
            shuffle=True,
        )
        print(f"\nPyG DataLoader created: {len(pyg_loader)} batches")
        print("PyTorch Geometric backend available for advanced training")
    except:
        print("\nPyTorch Geometric not installed")
        print("Install with: pip install weather-model-graphs[pytorch]")

except Exception as e:
    print(f"DataLoader creation note: {e}")

## Part 5: Backend Selection for Training

Choose the right backend for your use case:

In [ ]:
# Get backend for a graph
backend = wmg.get_backend(graph)
print(f"Current backend: {type(backend).__name__}")

# Extract features for model input
edge_index = backend.get_edge_index()
print(f"Edge index shape: {edge_index.shape}")

# Try PyTorch Geometric conversion
try:
    pyg_data = backend.to_pyg()
    if pyg_data is not None:
        print(f"\nPyTorch Geometric conversion successful:")
        print(f"  Nodes: {pyg_data.num_nodes}")
        print(f"  Edges: {pyg_data.num_edges}")
        print(f"  Edge index shape: {pyg_data.edge_index.shape}")
except Exception as e:
    print(f"PyG conversion note: {e}")

# Try DGL conversion
try:
    dgl_graph_obj = backend.to_dgl()
    if dgl_graph_obj is not None:
        print(f"\nDGL conversion successful:")
        print(f"  Nodes: {dgl_graph_obj.num_nodes()}")
        print(f"  Edges: {dgl_graph_obj.num_edges()}")
except Exception as e:
    print(f"DGL conversion note: {e}")

print("\n" + "="*60)
print("Backend Recommendation:")
print("="*60)
print("\n✓ NetworkX: Best for debugging, analysis, visualization")
print("✓ PyG: Best for general message-passing neural networks")
print("✓ DGL: Best for production, large-scale training")

## Part 6: Complete Training Pipeline Example

In [ ]:
# Pseudocode for complete pipeline
training_code = """
import weather_model_graphs as wmg
import torch
from torch.nn import Module, Linear
from torch_geometric.nn import GraphConv

# 1. DATA PREPARATION
graph = wmg.load_prebuilt('graphcast', grid_size=32)
for node in graph.nodes():
    # Add your weather data here
    graph.nodes[node]['features'] = [...]

# 2. FEATURE ENGINEERING
graph = wmg.add_wind_velocity(graph, u_attr='u', v_attr='v')
graph = wmg.normalize_features(graph, method='zscore')

# 3. DATA SPLIT
train_nodes, val_nodes, test_nodes = wmg.split_graph_for_training(graph)

# 4. CONVERT TO ML BACKEND
backend = wmg.get_backend(graph)
pyg_data = backend.to_pyg()

# 5. CREATE DATALOADERS
train_loader = wmg.create_pyg_dataloader([graph], batch_size=4)
val_loader = wmg.create_pyg_dataloader([graph], batch_size=4)

# 6. DEFINE AND TRAIN MODEL
class WeatherGNN(Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GraphConv(in_channels=256, out_channels=64)
        self.conv2 = GraphConv(in_channels=64, out_channels=64)
        self.out = Linear(64, 1)
    
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        return self.out(x)

model = WeatherGNN()
optimizer = torch.optim.Adam(model.parameters())
loss_fn = torch.nn.MSELoss()

# Training loop
for epoch in range(100):
    for batch in train_loader:
        y_pred = model(batch.x, batch.edge_index)
        loss = loss_fn(y_pred, batch.y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
"""

print(training_code)

## Summary: ML Pipeline Workflow

```
┌─────────────────────────────────┐
 │ 1. Load/Create Graph            │
 │   - Prebuilt architectures      │
 │   - Config-driven               │
 │   - Custom API                  │
 └──────────────┬──────────────────┘
                │
 ┌──────────────▼──────────────────┐
 │ 2. Add Features                 │
 │   - Weather variables (T, u, v) │
 │   - Engineered features         │
 │   - Normalization               │
 └──────────────┬──────────────────┘
                │
 ┌──────────────▼──────────────────┐
 │ 3. Data Splits                  │
 │   - Train/val/test (70/15/15)   │
 │   - Node-level or graph-level   │
 │   - Stratified sampling         │
 └──────────────┬──────────────────┘
                │
 ┌──────────────▼──────────────────┐
 │ 4. Backend Selection            │
 │   - NetworkX (debug)            │
 │   - PyG (training)              │
 │   - DGL (production)            │
 └──────────────┬──────────────────┘
                │
 ┌──────────────▼──────────────────┐
 │ 5. Create DataLoaders           │
 │   - Batching                    │
 │   - Shuffling                   │
 │   - Collation                   │
 └──────────────┬──────────────────┘
                │
 ┌──────────────▼──────────────────┐
 │ 6. Model Training               │
 │   - Graph neural network        │
 │   - Loss function               │
 │   - Optimization                │
 └─────────────────────────────────┘
```

**Key Functions**:
- `wmg.load_prebuilt()` - Load architecture
- `wmg.add_*()` - Add features
- `wmg.normalize_features()` - Normalize
- `wmg.split_graph_for_training()` - Data split
- `wmg.get_backend()` - Select format
- `wmg.create_dataloader()` - Create batches
- `wmg.batch_graphs()` - Combine graphs